In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from datetime import datetime
import http.client
import json
import os

In [2]:
load_dotenv()
api = os.getenv('OPENAI_KEY')

In [3]:
@tool
def fetch_exchange_rate(base='USD',target='INR'):
    """ This tool is  used to find the exchange value of the target currency for the given base currency """
    conn = http.client.HTTPSConnection("currency-conversion-and-exchange-rates.p.rapidapi.com")
    headers = {
        'x-rapidapi-key': "75d3aaafbcmshecf4cc8f0487295p1f1c76jsna54a9fdef709",
        'x-rapidapi-host': "currency-conversion-and-exchange-rates.p.rapidapi.com",
        'Content-Type': "application/json"
    }
    current_date = datetime.now()
    formatted_date = f"{current_date.year}-{current_date.month}-{current_date.day}"
    conn.request("GET", f"""/timeseries?start_date=2019-01-01&end_date=2019-01-01&base={base}&symbols={target}%2CGBP""", headers=headers)

    res = conn.getresponse()
    data = json.loads(res.read().decode())

    rate = data["rates"]["2019-01-01"][target]

    return rate



In [4]:
@tool
def exchange_amount(base_currency_value:int, conversion_rate:float) -> float:
    """ this tool returns the total value of the target currency against the value of the base currency with the given exchange rate"""
    return base_currency_value*conversion_rate

In [ ]:
#fetch_exchange_data('2019-01-01','USD','INR')

In [5]:
llm = ChatOpenAI(
    model = 'gpt-4.1-nano',
    api_key = api,
    temperature =0
)

In [6]:
llm_with_tools = llm.bind_tools([fetch_exchange_rate, exchange_amount])

In [ ]:
#fetch_exchange_data.args
#response = llm_with_tools.invoke('find the exchange rate for the INR against USD for 26-7-2026')
# result = response.tool_calls[0]["args"]

In [7]:
messages = []
system_prompt = SystemMessage('for calling the fetch_exchange_rate tool if the data is not dspecied then take the current date as pass it int the format yyyy-dd-mm')
#human_message = HumanMessage('How much inr will i get against 10000 usd')

#if not messages or not isinstance(messages[0], SystemMessage):
#        messages = [system_prompt] + messages

#messages.append(human_message)

#result = llm_with_tools.invoke(messages)
#print(result.tool_calls)

In [8]:
messages = [
    system_prompt,
    HumanMessage(
        content='how much inr i will get for 5000000 usd'
    )
]

while True:

    ai_message = llm_with_tools.invoke(messages)
    print(ai_message)
    messages.append(ai_message)

    if not ai_message.tool_calls:
        print(ai_message.content)
        break

    for tool_call in ai_message.tool_calls:

        if tool_call["name"] == "fetch_exchange_rate":
            result = fetch_exchange_rate.invoke(tool_call)

        elif tool_call["name"] == "exchange_amount":
            result = exchange_amount.invoke(tool_call)

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 163, 'total_tokens': 183, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_28f6c54d57', 'id': 'chatcmpl-E7zbz33f5U93qnvIeMZLRyv2lQQpE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019fbc7e-874d-7fa2-96a5-d15bafc8486f-0' tool_calls=[{'name': 'fetch_exchange_rate', 'args': {'base': 'USD', 'target': 'INR'}, 'id': 'call_8YYrTOuQLWbzkNnK5OnyOCJk', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 163, 'output_tokens': 20, 'total_tokens': 183, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
cont